# AXE 2 — 03. Movie Recommendations (MovieLens 32M)
_Notebook 3/3 — Docker cluster (phase 01 mode — données volumineuses)_

## Prérequis
- `01_exploration/ingest_movielens.ipynb` doit avoir tourné → `warehouse/movielens_*/`
- `01_build_interactions.ipynb` doit avoir tourné → `warehouse/interactions.parquet`

## Stratégie
1. Charger les 32M notes MovieLens (vrais utilisateurs)
2. Matcher les vues Netflix personnelles → `movieId` MovieLens (via titre normalisé)
3. Injecter l'utilisateur personnel (userId = 0) dans la matrice globale
4. Entraîner ALS sur la matrice complète (cluster distribué)
5. Générer les top 50 films non vus pour userId=0
6. Joindre avec `movielens_links` → tmdbId disponible pour le dashboard

## Output
`warehouse/movie_recommendations.parquet`  
Colonnes : `movieId`, `title`, `genres`, `tmdbId`, `predicted_score`, `rank`

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("MyDigitalTwin - ALS MovieReco") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

import sys as _sys
_sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import WAREHOUSE

def read_table(name):
    return spark.read.parquet(os.path.join(WAREHOUSE, name))

spark.sparkContext.setCheckpointDir(os.path.join(WAREHOUSE, "checkpoints"))

required = ["movielens_movies", "movielens_ratings", "movielens_links", "netflix_views"]
for t in required:
    path = os.path.join(WAREHOUSE, t)
    assert os.path.exists(path), f"Table manquante : {t} — lance le notebook prérequis"
print("Tables requises : OK")

In [ ]:
# ── 1. CHARGEMENT MOVIELENS ───────────────────────────────────────────────────

movies  = read_table("movielens_movies")   # movieId, title, title_norm, genres
ratings = read_table("movielens_ratings")  # userId, movieId, rating, timestamp
links   = read_table("movielens_links")    # movieId, imdbId, tmdbId

# Pas de .cache() sur 32M de lignes — saturerait la RAM en local.
# Spark lit directement depuis Parquet (column pruning natif).

print(f"Films MovieLens  : {movies.count():,}")
print(f"Notes MovieLens  : {ratings.count():,}")

In [ ]:
# ── 2. VUES NETFLIX PERSONNELLES → movieId MovieLens ─────────────────────────
# Normalisation : lowercase + suppression ponctuation (Column expressions, pas de UDF)
# Jointure sur title_norm déjà calculé lors de l'ingestion MovieLens.

netflix_raw = read_table("netflix_views").select(
    F.col("show_title").alias("netflix_title"),
    F.col("content_type")
).filter(
    F.lower(F.col("content_type")) == "movie"
).distinct()

netflix_norm = netflix_raw.withColumn(
    "title_norm",
    F.trim(F.regexp_replace(F.lower(F.col("netflix_title")), r'[^\w\s]', ''))
)

netflix_matched = netflix_norm.join(
    movies.select("movieId", "title", "title_norm"),
    on="title_norm",
    how="inner"
).select("movieId", "netflix_title", "title")

n_matched = netflix_matched.count()
n_total   = netflix_norm.count()

if n_total == 0:
    print("Aucun film trouvé — valeurs content_type disponibles :")
    read_table("netflix_views").groupBy("content_type").count().show()
else:
    print(f"Films Netflix matchés : {n_matched} / {n_total} ({100*n_matched//n_total}%)")
    netflix_matched.show(10, truncate=50)

In [ ]:
# ── 3. INTERACTIONS PERSONNELLES ─────────────────────────────────────────────
# userId = 0 (réservé — MovieLens commence à 1)
# Films vus plusieurs fois → rating légèrement boosté (max 5.0)

PERSONAL_USER_ID = 0
PERSONAL_RATING  = 4.5

netflix_counts = read_table("netflix_views") \
    .filter(F.lower(F.col("content_type")) == "movie") \
    .withColumn(
        "title_norm",
        F.trim(F.regexp_replace(F.lower(F.col("show_title")), r'[^\w\s]', ''))
    ) \
    .groupBy("title_norm") \
    .agg(F.count("*").alias("view_count"))

personal = netflix_matched \
    .withColumn(
        "title_norm",
        F.trim(F.regexp_replace(F.lower(F.col("netflix_title")), r'[^\w\s]', ''))
    ) \
    .join(netflix_counts, on="title_norm", how="left") \
    .select(
        F.lit(PERSONAL_USER_ID).cast(IntegerType()).alias("userId"),
        F.col("movieId"),
        F.least(
            F.lit(5.0),
            (F.lit(PERSONAL_RATING) + F.log1p(F.col("view_count").cast(FloatType())) * 0.1)
        ).alias("rating")
    )

print(f"Interactions personnelles : {personal.count()}")
personal.orderBy(F.desc("rating")).show(10)

In [ ]:
# ── 4. MATRICE COMBINÉE — Ultra-Densification (Survival Mode) ────────────────
from pyspark.sql.functions import broadcast

# On vise ~10-15M de lignes maximum pour 8GB de RAM
pop_movies = ratings.groupBy("movieId").agg(F.count("*").alias("n_r")).filter(F.col("n_r") >= 1000).select("movieId")
# On ne garde que les très gros parieurs (signal ultra-dense)
pwr_users = ratings.groupBy("userId").agg(F.count("*").alias("n_u")).filter(F.col("n_u") >= 200).select("userId")

ml_ratings = ratings \
 .join(broadcast(pop_movies), on="movieId", how="inner") \
 .join(broadcast(pwr_users), on="userId", how="inner") \
 .select(F.col("userId").cast("int"), F.col("movieId").cast("int"), F.col("rating").cast("float"))

combined_raw = ml_ratings.union(personal)

tmp_path = os.path.join(WAREHOUSE, "tmp_als_matrix")
print(f"Écriture de la matrice ultra-dense...")
combined_raw.write.mode("overwrite").parquet(tmp_path)

combined = spark.read.parquet(tmp_path).cache()
print(f"Matrice prête : {combined.count():,} interactions.")

In [ ]:
# ── 5. ENTRAÎNEMENT ALS (Lightweight) ─────────────────────────────────────────
als = ALS(
   userCol="userId",
   itemCol="movieId",
   ratingCol="rating",
   rank=8,           # Réduit pour économiser la RAM
   maxIter=10,       # Réduit pour éviter l'explosion du DAG
   regParam=0.1,
   implicitPrefs=False,
   coldStartStrategy="drop",
   checkpointInterval=2, # Checkpoint plus fréquent pour vider la RAM
   seed=42
)

print("Entraînement ALS Survival Mode...")
model = als.fit(combined)
print("Entraînement terminé !")

In [ ]:
# ── 6. TOP 50 RECOMMANDATIONS pour l'utilisateur personnel ───────────────────

personal_df = spark.createDataFrame(
    [(PERSONAL_USER_ID,)], ["userId"]
)

# Films déjà vus → à exclure
seen_ids = personal.select("movieId").distinct()

reco_raw = model.recommendForUserSubset(personal_df, 200) \
    .selectExpr("explode(recommendations) as rec") \
    .select(
        F.col("rec.movieId").cast(IntegerType()).alias("movieId"),
        F.col("rec.rating").cast(FloatType()).alias("predicted_score"),
    )

# Filtrer les films déjà vus
reco_unseen = reco_raw.join(seen_ids, on="movieId", how="left_anti")

print(f"Recommandations (non vus) : {reco_unseen.count()}")

In [ ]:
# ── 7. ENRICHISSEMENT — titre + genres + tmdbId + score percentile ────────────
# percent_rank() : évite un .count() séparé (qui forcerait un job Spark complet).
# Ordonnancement ascendant → worst=0.0, best=1.0 → ×100 donne 0-100%.

from pyspark.sql.window import Window

w_asc   = Window.orderBy(F.asc("predicted_score"))   # percent_rank : worst→best
w_rank  = Window.orderBy(F.desc("predicted_score"))  # row_number  : best first

reco_scored = reco_unseen \
    .withColumn("score_norm", F.round(F.percent_rank().over(w_asc) * 100, 1)) \
    .withColumn("row_num",    F.row_number().over(w_rank))

# Joindre avec movies (titre, genres) et links (tmdbId)
result = reco_scored \
    .join(movies.select("movieId", "title", "genres"), on="movieId", how="inner") \
    .join(links.select("movieId", "tmdbId"), on="movieId", how="left") \
    .select(
        "movieId",
        "title",
        "genres",
        "tmdbId",
        F.col("score_norm").alias("predicted_score"),
        F.col("row_num").alias("rank"),
    ) \
    .orderBy("rank") \
    .limit(50)

print("Top 10 recommandations :")
result.select("rank", "title", "predicted_score", "tmdbId").show(10, truncate=50)

In [ ]:
 # ── 8. ÉCRITURE warehouse/movie_recommendations/ ──────────────────────────────

out_path = os.path.join(WAREHOUSE, "movie_recommendations")
result.write.mode("overwrite").parquet(out_path)

# Vérification
check = spark.read.parquet(out_path)
print(f"Écrit : {out_path}")
print(f"Lignes : {check.count()}")
check.printSchema()

spark.stop()
print("\nNotebook 03 terminé. Les pages /recommandations et /netflix peuvent lire movie_recommendations.parquet.")